# 🔧 Módulo 03 — Function Tools

> **Objetivo:** Dar al agente acceso a herramientas (funciones Python) que puede invocar autónomamente.

## ¿Qué es un tool?

Un **tool** es una función Python decorada con `@tool` que el agente puede invocar cuando lo necesite.
El modelo decide qué tool llamar (y con qué argumentos) basándose en el contexto del mensaje.

```
Usuario: "¿Cómo está el clima en Madrid?"
   ↓
Agente: detecta que necesita get_weather
   ↓
Framework ejecuta get_weather(city="Madrid")
   ↓
Resultado: "Soleado, 28°C"
   ↓
Agente formula respuesta final usando ese resultado
```

## API en Python

```python
from agent_framework import tool
from typing import Annotated
from pydantic import Field

@tool(approval_mode="never_require")
def my_tool(param: Annotated[str, Field(description="...")]) -> str:
    """Descripción que ve el modelo."""
    return "..."
```

> `approval_mode="never_require"` ejecuta el tool sin pedir confirmación. El Módulo 04 verá `"always_require"`.


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Definir un tool simple

Las anotaciones `Annotated[type, Field(description=...)]` describen cada parámetro al modelo.
El docstring se usa como descripción del tool.


In [ ]:
from typing import Annotated
from agent_framework import tool
from pydantic import Field


@tool(approval_mode="never_require")
def get_weather(
    city: Annotated[str, Field(description="El nombre de la ciudad para obtener el clima")],
) -> str:
    """Obtiene el clima actual de una ciudad especificada."""
    weather = {
        "madrid": "Soleado, 28°C",
        "london": "Nublado, 15°C",
        "tokyo": "Lluvioso, 22°C",
        "tokio": "Lluvioso, 22°C",
        "new york": "Parcialmente nublado, 20°C",
    }
    info = weather.get(city.lower())
    return f"Clima en {city}: {info}" if info else f"Sin datos para {city}"


agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un asistente de clima. Usa la herramienta get_weather para responder "
        "preguntas sobre el clima. Responde en una oración."
    ),
    tools=[get_weather],
)

response = await agent.run("¿Cómo está el clima en Madrid?")
print(f"✅ {response.text}")


## 2️⃣ Múltiples tools — el modelo selecciona

Cuando el agente tiene varios tools, decide cuál usar según el mensaje del usuario.


In [ ]:
from datetime import datetime, timezone as tz


@tool(approval_mode="never_require")
def get_current_time(
    timezone: Annotated[str, Field(description="La zona horaria, p.ej. 'UTC', 'EST', 'CET'")],
) -> str:
    """Obtiene la hora actual en una zona horaria especificada."""
    now = datetime.now(tz.utc).strftime("%H:%M:%S")
    return f"Hora actual en {timezone}: {now} UTC (simulada)"


@tool(approval_mode="never_require")
def search_product(
    product_name: Annotated[str, Field(description="El nombre del producto a buscar")],
    max_results: Annotated[int, Field(description="Número máximo de resultados")] = 3,
) -> str:
    """Busca información de productos por nombre."""
    return (
        f"Encontrados {max_results} resultados para '{product_name}': "
        "Producto A ($29.99), Producto B ($49.99), Producto C ($19.99)"
    )


agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un asistente multi-propósito con acceso a herramientas de clima, hora "
        "y búsqueda de productos. Usa la apropiada según la pregunta. Sé conciso."
    ),
    tools=[get_weather, get_current_time, search_product],
)

print("✅ Probando 3 preguntas distintas (el agente elige el tool correcto):\n")
for q in ["¿Cómo está el clima en Tokio?", "¿Qué hora es en CET?", "Busca laptops"]:
    r = await agent.run(q)
    print(f"   • {q}\n     → {r.text}\n")


## 3️⃣ Plugin con estado compartido (smart-home)

Múltiples tools pueden compartir estado mediante variables del módulo.
Aquí simulamos un sistema de luces inteligentes con dos tools: `list_lights` y `toggle_light`.


In [ ]:
import json

_LIGHTS = [
    {"id": 1, "name": "Sala de estar", "is_on": True},
    {"id": 2, "name": "Cocina", "is_on": False},
    {"id": 3, "name": "Dormitorio", "is_on": False},
    {"id": 4, "name": "Oficina", "is_on": True},
]


@tool(approval_mode="never_require")
def list_lights() -> str:
    """Lista todas las luces inteligentes con su estado actual."""
    return json.dumps(_LIGHTS, ensure_ascii=False, indent=2)


@tool(approval_mode="never_require")
def toggle_light(
    light_id: Annotated[int, Field(description="El identificador numérico de la luz")],
    is_on: Annotated[bool, Field(description="True para encender, False para apagar")],
) -> str:
    """Enciende o apaga una luz inteligente específica."""
    for light in _LIGHTS:
        if light["id"] == light_id:
            light["is_on"] = is_on
            return f"Luz '{light['name']}' (Id: {light_id}) ahora está {'encendida' if is_on else 'apagada'}"
    return f"No se encontró una luz con Id {light_id}"


agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un asistente de hogar inteligente. Gestiona las luces usando list_lights "
        "y toggle_light. Responde siempre en español y de forma concisa."
    ),
    tools=[list_lights, toggle_light],
)

r1 = await agent.run("¿Cuáles luces tengo disponibles?")
print(f"✅ Listar: {r1.text}\n")

r2 = await agent.run("Enciende la luz de la cocina")
print(f"✅ Encender: {r2.text}\n")

r3 = await agent.run("Muéstrame el estado actual de todas las luces")
print(f"✅ Estado: {r3.text}")


## 🎯 Resumen

| Capacidad | Cómo se hace |
|-----------|--------------|
| Definir tool | `@tool(approval_mode="never_require")` sobre función con `Annotated[type, Field(description=...)]` |
| Dar tools al agente | `Agent(..., tools=[tool1, tool2, tool3])` |
| Estado compartido | Variables del módulo accedidas por varios tools |

**Siguiente módulo →** [Módulo 04 — Tool Approval (Human-in-the-Loop)](./04_tool_approval.ipynb)
